# MVP App
---

## Goal
Combine the RAG pipeline from Week 2 with situation profiling, a deadline calculator,
and a Gradio conversational UI to produce a working MVP demo.

## What's new vs NB  2
Situation-aware retrieval 
Deadline calculator
Conversational UI 


## Cell 1 — Install Dependencies

In [1]:
!pip install langchain langchain-groq langchain-community \
             chromadb fastembed gradio boto3 -q

## Cell 2 — Imports and Setup

Loads credentials from AWS Secrets Manager and connects to S3.
No API keys in code — all secrets are stored in Secrets Manager.

In [2]:
import os
import re
import json
import boto3
from datetime import datetime, date, timedelta

from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from fastembed import TextEmbedding
import chromadb
import gradio as gr

# Load Groq API key from AWS Secrets Manager
def get_secret(secret_name):
    client = boto3.client("secretsmanager", region_name="us-east-1")
    response = client.get_secret_value(SecretId=secret_name)
    return json.loads(response["SecretString"])

secrets = get_secret("immigration-navigator/groq")
os.environ["GROQ_API_KEY"] = secrets["GROQ_API_KEY"]

s3 = boto3.client("s3", region_name="us-east-1")
S3_BUCKET = "immigration-navigator-data"

print("Setup complete")

2026-06-05 21:44:33.811868933 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


✅ Setup complete


## Cell 3 — Load Models and Vector Store

Connects to the ChromaDB collection built in Week 2 (941 chunks).
No re-embedding needed — the vector store persists across sessions.

In [3]:
# Embedding model — same as Week 2
embedding_model = TextEmbedding("BAAI/bge-small-en-v1.5")
print("Embedding model loaded")

# Connect to existing ChromaDB vector store
chroma_client = chromadb.PersistentClient(path="/home/sagemaker-user/chroma_db")
collection = chroma_client.get_or_create_collection(
    name="immigration_navigator",
    metadata={"hnsw:space": "cosine"}
)
print(f"ChromaDB loaded — {collection.count()} chunks")

# LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=secrets["GROQ_API_KEY"],
    temperature=0
)
print("LLM ready")

Embedding model loaded


ChromaDB loaded — 941 chunks
LLM ready


## Cell 4 — Deadline Calculator

Deterministic rules-based module — no LLM involved.
All rules sourced from USCIS Policy Manual, Volume 2 Part F Chapter 5.

The LLM narrates; the rules engine computes.

In [4]:
def calculate_deadlines(graduation_date: date) -> dict:
    """
    Calculate key OPT, STEM OPT, and H-1B cap-gap deadlines
    based on the student's graduation date.

    All rules sourced from USCIS Policy Manual, Volume 2 Part F Chapter 5.

    Args:
        graduation_date (date): Student's program end date.

    Returns:
        dict: Named deadlines with dates and descriptions.
    """
    # OPT application window
    # No earlier than 90 days before graduation
    # No later than 60 days after graduation
    opt_app_earliest = graduation_date - timedelta(days=90)
    opt_app_latest   = graduation_date + timedelta(days=60)

    # OPT period — 12 months post-completion
    opt_end = graduation_date + timedelta(days=365)

    # STEM OPT — must apply 90 days before OPT expires, 24-month extension
    stem_app_deadline = opt_end - timedelta(days=90)
    stem_opt_end      = opt_end + timedelta(days=730)

    # H-1B timeline — approximate annual dates
    current_year = graduation_date.year
    h1b_lottery_open  = date(current_year, 3, 1)
    h1b_petition_date = date(current_year, 4, 1)
    h1b_start_date    = date(current_year, 10, 1)

    return {
        "opt_application": {
            "earliest": opt_app_earliest,
            "latest":   opt_app_latest,
            "note":     "File Form I-765 within this window"
        },
        "opt_period": {
            "start": graduation_date,
            "end":   opt_end,
            "note":  "12 months of OPT. Max 90 days unemployment."
        },
        "stem_opt_application": {
            "deadline": stem_app_deadline,
            "note":     "File Form I-765 + I-983. Must apply at least 90 days before OPT expires."
        },
        "stem_opt_period": {
            "start": opt_end,
            "end":   stem_opt_end,
            "note":  "24-month STEM OPT extension. Max 150 days unemployment."
        },
        "h1b_timeline": {
            "lottery_opens":  h1b_lottery_open,
            "petition_filed": h1b_petition_date,
            "start_date":     h1b_start_date,
            "note":           "Cap-gap automatically extends F-1 status until Oct 1 if H-1B is pending."
        }
    }


def format_deadlines(deadlines: dict) -> str:
    """Format deadlines dict as readable markdown string."""
    d = deadlines
    lines = [
        "📅 **Your Immigration Timeline**",
        "",
        "**OPT Application Window**",
        f"- Earliest: {d['opt_application']['earliest'].strftime('%B %d, %Y')}",
        f"- Latest:   {d['opt_application']['latest'].strftime('%B %d, %Y')}",
        f"- {d['opt_application']['note']}",
        "",
        "**OPT Period**",
        f"- Start: {d['opt_period']['start'].strftime('%B %d, %Y')}",
        f"- End:   {d['opt_period']['end'].strftime('%B %d, %Y')}",
        f"- {d['opt_period']['note']}",
        "",
        "**STEM OPT Extension**",
        f"- Apply by: {d['stem_opt_application']['deadline'].strftime('%B %d, %Y')}",
        f"- {d['stem_opt_application']['note']}",
        f"- Period: {d['stem_opt_period']['start'].strftime('%B %d, %Y')} to {d['stem_opt_period']['end'].strftime('%B %d, %Y')}",
        "",
        "**H-1B Timeline**",
        f"- Lottery opens:  {d['h1b_timeline']['lottery_opens'].strftime('%B %d, %Y')}",
        f"- Petition filed: {d['h1b_timeline']['petition_filed'].strftime('%B %d, %Y')}",
        f"- H-1B starts:   {d['h1b_timeline']['start_date'].strftime('%B %d, %Y')}",
        f"- {d['h1b_timeline']['note']}",
    ]
    return "\n".join(lines)


# Quick test
print(format_deadlines(calculate_deadlines(date(2025, 5, 15))))

📅 **Your Immigration Timeline**

**OPT Application Window**
- Earliest: February 14, 2025
- Latest:   July 14, 2025
- File Form I-765 within this window

**OPT Period**
- Start: May 15, 2025
- End:   May 15, 2026
- 12 months of OPT. Max 90 days unemployment.

**STEM OPT Extension**
- Apply by: February 14, 2026
- File Form I-765 + I-983. Must apply at least 90 days before OPT expires.
- Period: May 15, 2026 to May 14, 2028

**H-1B Timeline**
- Lottery opens:  March 01, 2025
- Petition filed: April 01, 2025
- H-1B starts:   October 01, 2025
- Cap-gap automatically extends F-1 status until Oct 1 if H-1B is pending.


## Cell 5 — RAG Pipeline with Situation-Aware Retrieval

The `ask()` function injects the user's profile into the query before
hitting the vector store. This means the same question gets a different
answer depending on visa stage, degree field, and employer type.

The LLM is prompted to cite every claim and decline if context is insufficient.

In [6]:
PROMPT = ChatPromptTemplate.from_template(
    """
You are ImmigrationNavigator, an AI assistant helping international students
navigate U.S. visa processes. Answer ONLY using the provided context.
Cite every claim with [Source: label].
If context is insufficient, say:
"I don't have enough information. Please consult your ISO or an immigration attorney."

User Profile:
- Visa status: {visa_status}
- Degree field: {degree_field}
- Graduation date: {graduation_date}
- Employer type: {employer_type}

Context:
{context}

Question: {question}

Answer (personalized to the user's profile, cite every claim):
"""
)


def ask(question: str, profile: dict, n_results: int = 5) -> str:
    """
    Full RAG pipeline with situation-aware retrieval.

    Injects the user's profile into the query before vector search
    so that retrieval is conditioned on their specific situation.

    Args:
        question  (str) : User's natural language question.
        profile   (dict): visa_status, degree_field, graduation_date, employer_type.
        n_results (int) : Number of chunks to retrieve.

    Returns:
        str: Cited, personalized answer from the LLM.
    """
    # Step 1 — Enrich query with user profile
    enriched_query = (
        f"{question} | "
        f"visa: {profile.get('visa_status','')} | "
        f"degree: {profile.get('degree_field','')} | "
        f"employer: {profile.get('employer_type','')}"
    )

    # Step 2 — Embed and retrieve top-k chunks
    query_embedding = list(embedding_model.embed([enriched_query]))[0].tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas"]
    )

    # Step 3 — Build context string with source metadata
    context_parts = [
        f"[Source: {meta['label']}, {meta['url']}]\n{doc}"
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]
    context = "\n\n".join(context_parts)

    # Step 4 — Generate cited answer
    chain = PROMPT | llm
    response = chain.invoke({
        "context":         context,
        "question":        question,
        "visa_status":     profile.get("visa_status",     "F-1"),
        "degree_field":    profile.get("degree_field",    "Not specified"),
        "graduation_date": profile.get("graduation_date", "Not specified"),
        "employer_type":   profile.get("employer_type",   "Not specified"),
    })
    return response.content


print("RAG pipeline ready")

RAG pipeline ready


## Cell 6 — Gradio App

Launches the conversational UI. Users set their profile in the sidebar
and ask questions in natural language.

When a question involves dates or deadlines, the deadline calculator
output is automatically appended to the LLM response.

Run this cell to get the public URL for user testing.

In [7]:
DEADLINE_KEYWORDS = ["deadline", "when", "date", "apply", "timeline", "expire", "window"]


def chat(message, history, visa_status, degree_field, graduation_date, employer_type):
    """
    Main chat function for Gradio.
    Builds the user profile from sidebar inputs, calls the RAG pipeline,
    and appends deadline calculator output when relevant.
    """
    profile = {
        "visa_status":     visa_status,
        "degree_field":    degree_field,
        "graduation_date": graduation_date,
        "employer_type":   employer_type,
    }

    answer = ask(message, profile)

    # Append deadline calculator if question involves dates
    if any(kw in message.lower() for kw in DEADLINE_KEYWORDS) and graduation_date:
        try:
            grad_date = datetime.strptime(graduation_date, "%Y-%m-%d").date()
            deadlines = calculate_deadlines(grad_date)
            answer += "\n\n---\n" + format_deadlines(deadlines)
        except ValueError:
            pass

    return answer


with gr.Blocks(title="ImmigrationNavigator") as app:
    gr.Markdown("""
    # 🧭 ImmigrationNavigator
    **AI-powered guidance for the F-1 → OPT → STEM OPT → H-1B pipeline**

    > ⚠️ This tool provides guidance based on official USCIS sources.
    > It is **not legal advice**. For complex situations, consult an immigration attorney or your ISO.
    """)

    gr.ChatInterface(
        fn=chat,
        additional_inputs=[
            gr.Dropdown(
                choices=["F-1 student", "F-1, currently on OPT",
                         "F-1, on STEM OPT", "H-1B pending"],
                label="Current visa status",
                value="F-1 student"
            ),
            gr.Dropdown(
                choices=["Computer Science (STEM)", "Data Science (STEM)",
                         "Engineering (STEM)", "Biology (STEM)",
                         "Business (non-STEM)", "Humanities (non-STEM)", "Other"],
                label="Degree field",
                value="Computer Science (STEM)"
            ),
            gr.Textbox(
                label="Graduation date (YYYY-MM-DD)",
                placeholder="e.g. 2025-05-15",
                value=""
            ),
            gr.Dropdown(
                choices=["Full-time employer", "Part-time employer",
                         "Multiple employers", "Consulting firm",
                         "Self-employed", "Not yet employed"],
                label="Employer type",
                value="Full-time employer"
            ),
        ],
        examples=[
            ["When do I need to apply for OPT?"],
            ["Am I eligible for STEM OPT extension?"],
            ["What happens during the H-1B cap-gap period?"],
            ["What forms do I need for OPT?"],
            ["How many days can I be unemployed on OPT?"],
        ],
        title="",
    )

app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860


* Running on public URL: https://26877b0f6d42366848.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
